# Tratamiento de datos

Antes de hablar de probabilidad necesitamos **describir** los datos: ubicarlos,
cuantificar su dispersión y detectar puntos raros. Esta unidad arma el vocabulario
que el resto del libro va a usar como base.

Imaginemos tres escenarios cotidianos que vamos a volver a encontrar a lo largo
del libro: una sala de espera de una clínica donde se anota cuánto esperó cada
paciente, una encuesta electoral con una muestra de personas que dicen sí o no, y
una línea de producción donde un turno entero se inspecciona pieza por pieza.
Los tres son fuentes de datos pero plantean preguntas distintas. En esta unidad
nos concentramos en **describir** lo observado; todavía no hablamos de
probabilidades.


## Preguntas de inicio

1. ¿Cómo resumimos en pocos números los tiempos de espera de la clínica?
2. ¿Cómo detectamos un día raro: un paciente que esperó muchísimo, un turno con
   muchas piezas defectuosas?
3. ¿Por qué a veces la **media** miente y conviene mirar la **mediana**?
4. ¿Cómo comparamos dispersiones de muestras con escalas distintas (minutos vs.
   piezas)?

Vamos a responder cada una al final, en la sección **Síntesis y respuestas**.


## Importaciones

In [ ]:
import numpy as np
import pandas as pd

from core import Observations, Settings
from descriptive import (
    FrequencyTableInput,
    build_frequency_table,
    detect_outliers_tukey,
    standardize_observations,
    summarize_observations,
)
from exercises import NumericAnswerInput, verify_numeric_answer
from visualization import (
    DescriptiveSummaryChartInput,
    FrequencyChartInput,
    HistogramChartInput,
    chart_descriptive_summary,
    chart_frequency_table,
    chart_histogram,
)
from widgets import DescriptiveExplorerInput, build_descriptive_explorer

In [ ]:
settings = Settings()

## Una muestra de tiempos de espera

Registramos los minutos que esperaron 80 pacientes en una mañana en la clínica.
Para que el ejemplo sea reproducible, los simulamos a partir de un mecanismo
centrado en 4 minutos con cierta variabilidad. La unidad de **probabilidad** que
justifica este mecanismo aparece recién en el capítulo siguiente; por ahora la
tratamos como una muestra concreta.


In [ ]:
rng_clinic = np.random.default_rng(settings.random_seed)
raw_waiting_times = rng_clinic.normal(loc=4.0, scale=1.2, size=80).clip(min=0.0)
waiting_times = Observations.validate(pd.DataFrame({"value": raw_waiting_times}))
waiting_times.head()

Antes de cualquier número, **dibujamos**. Construimos una tabla de frecuencias y
la usamos para mostrar dos cosas a la vez: el histograma (barras, frecuencia
absoluta) y la **ojiva** (línea, frecuencia relativa acumulada).


In [ ]:
frequency_input = FrequencyTableInput(observations=waiting_times, bin_count=10)
frequency_table = build_frequency_table(frequency_input)

frequency_chart_input = FrequencyChartInput(
    frequency_table=frequency_table,
    title="Tiempos de espera (clínica) — histograma y ojiva",
    settings=settings,
)
chart_frequency_table(frequency_chart_input)

## Resumir con tres miradas: posición, dispersión y outliers

La **media muestral** $\bar{x}$ aparece como pregunta natural: ¿cuál es el centro
de gravedad de la muestra? Con $n$ observaciones $x_1, \dots, x_n$:

$$ \bar{x} = \frac{1}{n}\sum_{i=1}^{n} x_i \tag{1.1} $$

La **mediana** $\tilde{x}$ es el valor que parte la muestra ordenada por la mitad.
Para $n$ par se promedia el par central:

$$ \tilde{x} = \begin{cases}
  x_{((n+1)/2)} & \text{si } n \text{ es impar} \\
  \dfrac{x_{(n/2)} + x_{(n/2+1)}}{2} & \text{si } n \text{ es par}
\end{cases} \tag{1.2} $$

El **desvío estándar muestral** $s$ mide cuánto se separan, en promedio, las
observaciones respecto de $\bar{x}$. Se define paso a paso. **Paso 1:** partimos
de la fórmula (1.1) para tener el centro:

$$ \bar{x} = \frac{1}{n}\sum_{i=1}^{n} x_i $$

**Paso 2:** sumamos los desvíos al cuadrado (cuadrar elimina los signos):

$$ \text{SS} = \sum_{i=1}^{n}(x_i - \bar{x})^2 $$

**Paso 3:** dividimos por $n-1$ y tomamos raíz para volver a la unidad original:

$$ s = \sqrt{\frac{\text{SS}}{n - 1}} \tag{1.3} $$


In [ ]:
summary = summarize_observations(waiting_times)
summary

Tres preguntas distintas conviven en cualquier muestra y la tabla de arriba
responde a las tres a la vez: ¿dónde se sitúa el centro de la nube de
observaciones?, ¿qué tan apretadas viven las observaciones alrededor de ese
centro?, ¿hay puntos que se desvían lo suficiente del resto como para
tratarlos aparte? Pasá la vista por las filas y fijate qué números te
convencen sobre cada una. La media y la mediana hablan del centro; el
desvío estándar y el rango intercuartil, del ancho; las cotas inferior y
superior delimitan lo que para esta muestra cuenta como observación habitual.


## Por qué la mediana resiste lo que la media no

Cuando la distribución es **simétrica**, $\bar{x}$ y $\tilde{x}$ están casi
pegados. Cuando aparecen valores muy altos (un paciente con espera anormal) la
**media se mueve** hacia ese punto; la **mediana se queda quieta** porque sólo
depende del orden.

Lo materializamos con un boxplot que marca, además, la media muestral con una
línea punteada: cuando hay cola, las dos referencias se separan.


In [ ]:
descriptive_chart_input = DescriptiveSummaryChartInput(
    observations=waiting_times,
    statistics=summary,
    title="Tiempos de espera — boxplot con media muestral",
    settings=settings,
)
chart_descriptive_summary(descriptive_chart_input)

## Detección de outliers — la regla de Tukey

**Paso 1:** partimos de los cuartiles $Q_1$ y $Q_3$ que ya calculó la fórmula
(1.2) en su versión generalizada. **Paso 2:** definimos el rango intercuartil
$\text{IQR} = Q_3 - Q_1$. **Paso 3:** una observación es **outlier** si cae
fuera del intervalo:

$$ \bigl[Q_1 - 1{,}5\,\text{IQR},\ Q_3 + 1{,}5\,\text{IQR}\bigr] \tag{1.4} $$

Es la misma regla que usa la caja del boxplot: por eso ese gráfico es buen
detector visual.


In [ ]:
outlier_report = detect_outliers_tukey(waiting_times)
outlier_report

## Posición relativa: el $z$-score

El $z$-score traduce una observación a la pregunta «¿a cuántos desvíos del
promedio está?». **Paso 1:** partimos de las fórmulas (1.1) y (1.3) para tener
$\bar{x}$ y $s$. **Paso 2:** centramos restando el promedio. **Paso 3:**
reescalamos dividiendo por $s$:

$$ z_i = \frac{x_i - \bar{x}}{s} \tag{1.5} $$

Esta misma transformación va a reaparecer en el capítulo de variables
aleatorias (ecuación 3.1), pero allá con un significado más fuerte: convertir
una Normal cualquiera en la Normal estándar.


In [ ]:
standardized = standardize_observations(waiting_times)
standardized

## Piezas defectuosas por turno

Cambiamos de escala y de unidad. La línea de producción registra, en cada uno
de los últimos 60 turnos, cuántas piezas resultaron defectuosas tras la
inspección. Acá no estamos midiendo en minutos: estamos contando.


In [ ]:
rng_factory = np.random.default_rng(seed=20260202)
raw_defect_counts = rng_factory.poisson(lam=3.5, size=60).astype(float)
defect_counts = Observations.validate(pd.DataFrame({"value": raw_defect_counts}))

defect_histogram_input = HistogramChartInput(
    observations=defect_counts,
    bin_count=12,
    title="Piezas defectuosas por turno (60 turnos)",
    settings=settings,
)
chart_histogram(defect_histogram_input)

In [ ]:
defect_summary = summarize_observations(defect_counts)
defect_summary

Las dos muestras viven en escalas distintas — minutos contra piezas — y con
magnitudes muy distintas. Comparar sus medias en crudo dice poco: 4 minutos
no es ni mejor ni peor que 3 piezas defectuosas. Cuando la comparación se
vuelve incómoda hay que salir de las unidades originales: los $z$-scores,
que ya introdujimos, y el **coeficiente de variación** $s/\bar{x}$ son
adimensionales y permiten comparar sin trampas. ¿Cuál de las dos muestras
tiene, relativo a su propia media, más dispersión?


## Exploración interactiva

Probá esa intuición moviendo $\sigma$ con el control de abajo. ¿Qué crece
cuando crece la dispersión, y qué se queda quieto? La caja del boxplot debería
responder a ese cambio; la mediana, no.


In [ ]:
explorer_input = DescriptiveExplorerInput(settings=settings)
build_descriptive_explorer(explorer_input)

## Ejercicio 1 — Media de una muestra pequeña

Tomá la muestra $\{2, 4, 4, 4, 5, 5, 7, 9\}$ y calculá la media muestral
aplicando la fórmula (1.1).

**Idea.** Sumar y dividir por $n = 8$:

$$ \bar{x} = \frac{2 + 4 + 4 + 4 + 5 + 5 + 7 + 9}{8} = \frac{40}{8} = 5 $$


In [ ]:
exercise_sample = Observations.validate(pd.DataFrame({"value": [2.0, 4.0, 4.0, 4.0, 5.0, 5.0, 7.0, 9.0]}))
expected_mean = summarize_observations(exercise_sample).location.mean

student_answer_mean = 5.0
verify_input_mean = NumericAnswerInput(
    student_answer=student_answer_mean,
    expected_answer=expected_mean,
)
verify_numeric_answer(verify_input_mean)

## Ejercicio 2 — Desvío estándar muestral

Para la misma muestra, aplicá la fórmula (1.3). Con $\bar{x} = 5$ del
ejercicio 1:

$$ \begin{aligned}
\text{SS} &= (2-5)^2 + 3(4-5)^2 + 2(5-5)^2 + (7-5)^2 + (9-5)^2 \\
           &= 9 + 3 + 0 + 4 + 16 \\
           &= 32 \\
s          &= \sqrt{\dfrac{32}{7}} \approx 2{,}138
\end{aligned} $$


In [ ]:
expected_std = summarize_observations(exercise_sample).dispersion.sample_standard_deviation

student_answer_std = 2.138
verify_input_std = NumericAnswerInput(
    student_answer=student_answer_std,
    expected_answer=expected_std,
    absolute_tolerance=1e-2,
    relative_tolerance=1e-2,
)
verify_numeric_answer(verify_input_std)

## Síntesis y respuestas

1. **¿Cómo resumimos los tiempos de espera?** Con tres miradas
   complementarias: la posición ubica el centro de la muestra, la dispersión
   cuantifica el ancho de la nube alrededor de ese centro y el reporte de
   outliers separa lo habitual de lo raro. Una sola es siempre insuficiente.
2. **¿Cómo detectamos un día raro?** Cualquier observación fuera del intervalo
   $[Q_1 - 1{,}5\,\text{IQR},\ Q_3 + 1{,}5\,\text{IQR}]$ — la regla de Tukey de
   la ecuación (1.4) — queda señalada como candidata a outlier.
3. **¿Cuándo miente la media?** Cuando la muestra tiene cola larga o algún
   valor extremo. La mediana, definida en (1.2), no se inmuta: depende del
   orden, no de los valores.
4. **¿Cómo comparamos dispersiones de escalas distintas?** Saliendo de las
   unidades originales. Convirtiendo a $z$-scores (ecuación 1.5) o calculando
   el **coeficiente de variación** $s/\bar{x}$ tenemos números adimensionales
   que pueden compararse entre muestras de naturaleza muy distinta.


Lo descrito hasta acá es siempre lo que **ya pasó**: un puñado de mañanas en
la clínica, algunos turnos de la línea de producción. Es información valiosa
pero limitada. Apenas entra un paciente nuevo o llega un turno todavía no
inspeccionado, estos resúmenes dejan de alcanzar: ¿qué tan probable es que el
próximo paciente espere más de cinco minutos?, si ya lleva tres minutos
esperando, ¿cambian las chances?, cuando alguien dice «sí» en la encuesta,
¿qué tan creíble es esa respuesta dado lo que ya sabemos de la población?
Para responder hay que dejar de mirar muestras y empezar a hablar de futuros
posibles — un idioma con su propia gramática, la **probabilidad**.
